In [1]:
from igraph import read
import os

os.makedirs("../graph", exist_ok=True)
output_path = "../graph/boardgames_graph_full.graphml"

# When exporting to DOT, igraph will include vertex attributes like 'pos'
g = read(output_path, format="graphml")
print(f"Read DOT file with positions: {output_path} ({g.vcount()} nodes, {g.ecount()} edges)")
print(g.summary())

Read DOT file with positions: ../graph/boardgames_graph_full.graphml (40590 nodes, 328774 edges)
IGRAPH U-W- 40590 328774 -- 
+ attr: community (v), game_name (v), id (v), weight (e)


/opt/conda/envs/bgg-gt/lib/python3.10/site-packages/igraph/io/files.py:295: RuntimeWarning: Could not add vertex ids, there is already an 'id' vertex attribute. at src/io/graphml.c:485
  return reader(f, *args, **kwds)


In [2]:
# Community-biased TMFG: ensure each community subgraph is connected
import numpy as np
import igraph as ig
import importlib
import TMFG_core as _tmfgmod
from scipy import sparse as sp
_tmfgmod = importlib.reload(_tmfgmod)
# Debug: show which TMFG_core is loaded
try:
    print("TMFG_core loaded from:", _tmfgmod.__file__)
except Exception:
    pass
TMFG = _tmfgmod.TMFG

# 1) Build weights directly from current igraph 'g' as a sparse matrix to save memory

n = g.vcount()
if n < 4:
    raise RuntimeError("TMFG requires at least 4 vertices.")

rows, cols, vals = [], [], []
for e in g.es:
    w = float(e['weight']) if 'weight' in e.attributes() else 0.0
    i, j = e.source, e.target
    rows.append(i); cols.append(j); vals.append(w)
    rows.append(j); cols.append(i); vals.append(w)

# Correlation-like matrix setup (sparse)
# Diagonal set to 1.0 and add jitter without densifying
jitter = 1e-12
W_sparse = sp.csr_matrix((vals, (rows, cols)), shape=(n, n))
W_sparse.setdiag(1.0 + jitter)
cov_matrix = W_sparse.copy()

# 2) Community membership aligned to g's vertex order

comm_membership = np.array(g.vs['community'], dtype=int)

# 3) Run TMFG with community bias (sparse input, sparse output)
boost = 4.5      # increase if needed (e.g., 3.0, 4.0)
penalty = 0.4  # lower to discourage inter-community edges (e.g., 0.8)
print(f"Running community-biased TMFG (sparse): boost={boost}, penalty={penalty}")
model = TMFG()
model.fit(
    weights=W_sparse,
    output='unweighted_sparse_W_matrix',
    cov=cov_matrix,
    communities=comm_membership,
    intra_boost=boost,
    inter_penalty=penalty,
)
cliques_b, seps_b, J_b = model.transform()

# 4) Build igraph from TMFG adjacency (sparse)
tmfg_backbone_graph_biased = ig.Graph(n=n, directed=False)
# Transfer common vertex attributes when available
for attr in ('id', 'name', 'game_name'):
    if attr in g.vs.attributes():
        tmfg_backbone_graph_biased.vs[attr] = g.vs[attr]
# Communities
tmfg_backbone_graph_biased.vs['community'] = comm_membership.tolist()
# Edges from sparse adjacency (symmetry handled by i<j)
edges_to_add, weights_to_add = [], []
ri, rj = J_b.nonzero()
for i, j in zip(ri, rj):
    if i < j:
        edges_to_add.append((i, j))
        eid = g.get_eid(i, j, directed=False, error=False)
        w = float(g.es[eid]['weight']) if eid != -1 and 'weight' in g.es[eid].attributes() else 0.0
        weights_to_add.append(w)
if edges_to_add:
    tmfg_backbone_graph_biased.add_edges(edges_to_add)
    tmfg_backbone_graph_biased.es['weight'] = weights_to_add

print(f"Biased TMFG graph: {tmfg_backbone_graph_biased.vcount()} nodes, {tmfg_backbone_graph_biased.ecount()} edges")

# 5) Verify per-community induced subgraph connectivity
unique_comms = sorted(set(comm_membership.tolist()))
not_connected = []
for c in unique_comms:
    vs_idx = [v.index for v in tmfg_backbone_graph_biased.vs if v['community'] == c]
    if not vs_idx:
        continue
    subg = tmfg_backbone_graph_biased.subgraph(vs_idx)
    if subg.vcount() > 1 and not subg.is_connected():
        not_connected.append((c, subg.components().sizes()))
tmfg_backbone_graph=tmfg_backbone_graph_biased
if not not_connected:
    print(f"Success: All {len(unique_comms)} communities are connected with boost={boost}, penalty={penalty}.")
else:
    print(f"Communities not connected (boost={boost}, penalty={penalty}):")
    for c, sizes in not_connected:
        print(f"  community={c}, component sizes={list(sizes)}")
    print("Hint: increase 'boost' or reduce 'penalty' and re-run.")


TMFG_core loaded from: /workspaces/python-bgg/notebooks/TMFG_core.py
Running community-biased TMFG (sparse): boost=4.5, penalty=0.4
Biased TMFG graph: 40590 nodes, 121764 edges
Success: All 56 communities are connected with boost=4.5, penalty=0.4.


In [3]:
# Step 1: Remove zero-weight edges from tmfg_backbone_graph
import numpy as np

print("Original tmfg_backbone_graph:")
print(f"  Vertices: {tmfg_backbone_graph.vcount()}")
print(f"  Edges: {tmfg_backbone_graph.ecount()}")
print(f"  Connected: {tmfg_backbone_graph.is_connected()}")

# Create filtered graph without zero-weight edges
tmfg_filtered = tmfg_backbone_graph.copy()
zero_weight_edge_ids = [e.index for e in tmfg_filtered.es if e['weight'] == 0.0]
print(f"\nRemoving {len(zero_weight_edge_ids)} zero-weight edges...")
tmfg_filtered.delete_edges(zero_weight_edge_ids)

print(f"\nFiltered tmfg_backbone_graph:")
print(f"  Vertices: {tmfg_filtered.vcount()}")
print(f"  Edges: {tmfg_filtered.ecount()}")
print(f"  Connected: {tmfg_filtered.is_connected()}")

if not tmfg_filtered.is_connected():
    components = tmfg_filtered.connected_components()
    print(f"  Number of components: {len(components)}")
    print(f"  Component sizes: {sorted(components.sizes(), reverse=True)}")

Original tmfg_backbone_graph:
  Vertices: 40590
  Edges: 121764
  Connected: True

Removing 37071 zero-weight edges...

Filtered tmfg_backbone_graph:
  Vertices: 40590
  Edges: 84693
  Connected: False
  Number of components: 3
  Component sizes: [36342, 2149, 2099]


In [4]:
# Step 2: Find minimum edges from g to reconnect components (maximizing total weight)
# Strategy: For each pair of components, find the maximum weight edge between them from g

from igraph import Graph
import heapq

# Build id to vertex index maps
id_to_vertex_g = {v['id']: v.index for v in g.vs}
id_to_vertex_filtered = {v['id']: v.index for v in tmfg_filtered.vs}

# Get component membership
components = tmfg_filtered.connected_components()
membership = components.membership

print(f"Finding edges to reconnect {len(components)} components...")
print()

# For each pair of components, find all candidate edges from g
# and select the one with maximum weight
component_pairs = []
for i in range(len(components)):
    for j in range(i + 1, len(components)):
        # Get vertex IDs in each component
        comp_i_vertices = [tmfg_filtered.vs[v_idx]['id'] for v_idx in range(tmfg_filtered.vcount()) if membership[v_idx] == i]
        comp_j_vertices = [tmfg_filtered.vs[v_idx]['id'] for v_idx in range(tmfg_filtered.vcount()) if membership[v_idx] == j]
        
        # Find best edge between these components in g
        best_weight = -1
        best_edge = None
        edges_checked = 0
        
        for v_id_i in comp_i_vertices:
            v_idx_g_i = id_to_vertex_g.get(v_id_i)
            if v_idx_g_i is None:
                continue
                
            for neighbor_idx in g.neighbors(v_idx_g_i):
                neighbor_id = g.vs[neighbor_idx]['id']
                if neighbor_id in comp_j_vertices:
                    edges_checked += 1
                    edge_id = g.get_eid(v_idx_g_i, neighbor_idx, directed=False, error=False)
                    if edge_id != -1:
                        weight = g.es[edge_id]['weight'] if 'weight' in g.es[edge_id].attributes() else 0.0
                        if weight > best_weight:
                            best_weight = weight
                            best_edge = (v_id_i, neighbor_id, weight)
        
        if best_edge:
            component_pairs.append((i, j, best_weight, best_edge))
            print(f"Components {i} ↔ {j}: best edge weight = {best_weight:.6f} ({edges_checked} candidates)")

# Sort by weight (descending) to prioritize highest-weight connections
component_pairs.sort(key=lambda x: x[2], reverse=True)
print(f"\nFound {len(component_pairs)} potential inter-component edges")

Finding edges to reconnect 3 components...

Components 0 ↔ 1: best edge weight = 0.207729 (7234 candidates)
Components 0 ↔ 2: best edge weight = 0.221757 (7309 candidates)
Components 1 ↔ 2: best edge weight = 0.238095 (315 candidates)

Found 3 potential inter-component edges


In [5]:
# Step 3: Use greedy approach to add minimum edges that maximize total weight
# We need to connect n components with n-1 edges (minimum spanning tree approach)

# Use union-find to track component merging
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        root_x, root_y = self.find(x), self.find(y)
        if root_x == root_y:
            return False
        if self.rank[root_x] < self.rank[root_y]:
            self.parent[root_x] = root_y
        elif self.rank[root_x] > self.rank[root_y]:
            self.parent[root_y] = root_x
        else:
            self.parent[root_y] = root_x
            self.rank[root_x] += 1
        return True

uf = UnionFind(len(components))
edges_to_add = []
total_weight_added = 0.0

print("Selecting edges to reconnect components (maximizing total weight):")
print()

for comp_i, comp_j, weight, (v_id_i, v_id_j, w) in component_pairs:
    if uf.union(comp_i, comp_j):
        edges_to_add.append((v_id_i, v_id_j, w))
        total_weight_added += w
        v_name_i = None
        v_name_j = None
        for v in tmfg_filtered.vs:
            if v['id'] == v_id_i:
                v_name_i = v['game_name'] if 'game_name' in v.attributes() else str(v_id_i)
            if v['id'] == v_id_j:
                v_name_j = v['game_name'] if 'game_name' in v.attributes() else str(v_id_j)
        print(f"  Adding: {v_name_i} -- {v_name_j} (weight={w:.6f})")

print(f"\nTotal edges to add: {len(edges_to_add)}")
print(f"Total weight added: {total_weight_added:.6f}")

Selecting edges to reconnect components (maximizing total weight):

  Adding: Catena -- Complica (weight=0.238095)
  Adding: Dickory -- Lepidoptery (weight=0.221757)

Total edges to add: 2
Total weight added: 0.459853


In [6]:
# Step 4: Add selected edges to the filtered graph
print("Adding reconnection edges to tmfg_filtered...")

for v_id_i, v_id_j, weight in edges_to_add:
    # Find vertex indices in filtered graph
    v_idx_i = id_to_vertex_filtered[v_id_i]
    v_idx_j = id_to_vertex_filtered[v_id_j]
    
    # Add edge
    tmfg_filtered.add_edge(v_idx_i, v_idx_j, weight=weight)

print(f"\nFinal tmfg_filtered graph:")
print(f"  Vertices: {tmfg_filtered.vcount()}")
print(f"  Edges: {tmfg_filtered.ecount()}")
print(f"  Connected: {tmfg_filtered.is_connected()}")

# Verify all communities are still connected
unique_comms = sorted(set(tmfg_filtered.vs['community']))
not_connected = []
for c in unique_comms:
    vs_idx = [v.index for v in tmfg_filtered.vs if v['community'] == c]
    if len(vs_idx) > 1:
        subg = tmfg_filtered.subgraph(vs_idx)
        if not subg.is_connected():
            not_connected.append(c)

if not_connected:
    print(f"\n⚠️  Warning: {len(not_connected)} communities are not internally connected")
    print(f"  Communities: {not_connected[:10]}")
else:
    print(f"\n✓ All {len(unique_comms)} communities are internally connected")

# Update tmfg_backbone_graph to be the filtered version
tmfg_backbone_graph = tmfg_filtered
print(f"\n✓ tmfg_backbone_graph updated (zero-weight edges removed, reconnected with {len(edges_to_add)} high-weight edges)")

Adding reconnection edges to tmfg_filtered...

Final tmfg_filtered graph:
  Vertices: 40590
  Edges: 84695
  Connected: True

✓ All 56 communities are internally connected

✓ tmfg_backbone_graph updated (zero-weight edges removed, reconnected with 2 high-weight edges)


In [7]:
# Sum of edge weights in tmfg_backbone_graph
if 'weight' in tmfg_backbone_graph.es.attributes():
    total_weight = float(sum(e['weight'] for e in tmfg_backbone_graph.es))
    print(f"Total edge weight in tmfg_backbone_graph: {total_weight}")
else:
    print("Edges in tmfg_backbone_graph do not have a 'weight' attribute.")

Total edge weight in tmfg_backbone_graph: 7446.443212812967


In [8]:
print("Final tmfg_backbone_graph summary:")
print(tmfg_backbone_graph.summary())

Final tmfg_backbone_graph summary:
IGRAPH U-W- 40590 84695 -- 
+ attr: community (v), game_name (v), id (v), weight (e)


In [9]:
import os
from igraph import write


os.makedirs("../graph", exist_ok=True)
output_path = "../graph/boardgames_graph_tmfg.graphml"

# When exporting to DOT, igraph will include vertex attributes like 'pos'
write(tmfg_backbone_graph, output_path)
print(f"Written DOT file with positions: {output_path} ({tmfg_backbone_graph.vcount()} nodes, {tmfg_backbone_graph.ecount()} edges)")

Written DOT file with positions: ../graph/boardgames_graph_tmfg.graphml (40590 nodes, 84695 edges)


In [11]:
from igraph import read
import os

os.makedirs("../graph", exist_ok=True)
output_path = "../graph/boardgames_graph_tmfg.graphml"

# When exporting to DOT, igraph will include vertex attributes like 'pos'
tmfg_backbone_graph = read(output_path, format="graphml")
print(f"Read DOT file with positions: {output_path} ({tmfg_backbone_graph.vcount()} nodes, {tmfg_backbone_graph.ecount()} edges)")
print(tmfg_backbone_graph.summary())

Read DOT file with positions: ../graph/boardgames_graph_tmfg.graphml (40590 nodes, 84695 edges)
IGRAPH U-W- 40590 84695 -- 
+ attr: community (v), game_name (v), id (v), weight (e)


/opt/conda/envs/bgg-gt/lib/python3.10/site-packages/igraph/io/files.py:295: RuntimeWarning: Could not add vertex ids, there is already an 'id' vertex attribute. at src/io/graphml.c:485
  return reader(f, *args, **kwds)


In [3]:
import pydot
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import numpy as np
# ----------------------------
# Helpers
# ----------------------------
def build_id_to_name_map(graph):
    """Build a dictionary mapping node 'id' attribute to node name."""
    return {
        str(int(float(attrs.get("id")))): node.get_name()
        for node in graph.get_nodes()
        if (attrs := node.get_attributes()).get("id") is not None
    }

def process_one_graph(graph, g, backbone_graph, j):
    """Process one subgraph j and return extended pydot graph.

    Guarantees: all edges from tmfg_backbone_graph affecting this community
    appear with status == "Visible" (including cross-community duplicates).
    """

    # Precompute caches
    id_to_name_cache = build_id_to_name_map(graph)
    nodes = graph.get_nodes()
    node_attrs = {str(i): int(float(nodes[i].get_attributes()["id"])) for i in range(len(nodes))}

    # FIX: Build node_to_comm correctly from backbone_graph which has the community attribute
    node_to_comm = {v["id"]: v["community"] for v in backbone_graph.vs}

    # igraph: build id → vertex_index maps for O(1) access
    id_to_vertex_g = {v["id"]: v.index for v in g.vs}
    id_to_vertex_backbone = {v["id"]: v.index for v in backbone_graph.vs}

    # Precompute edges present in this pydot subgraph (for thresholds)
    min_weight = defaultdict(lambda: float("inf"))
    neighbor_map = defaultdict(set)
    for edge in graph.get_edges():
        w = float(edge.get_attributes().get("weight", float("inf")))
        u, v = edge.get_source(), edge.get_destination()
        if w < min_weight[u]: min_weight[u] = w
        if w < min_weight[v]: min_weight[v] = w
        neighbor_map[u].add(v)
        neighbor_map[v].add(u)

    # Process nodes (sequential inside this process)
    all_results = []
    for target_node, node_id in node_attrs.items():
        node_index_g = id_to_vertex_g[node_id]
        node_g = g.vs[node_index_g]

        comm_self = node_to_comm[node_id]
        edges_info = []

        # Compute safe threshold for this node
        node_min = min_weight[target_node]
        if not np.isfinite(node_min):
            node_min = 0.0  # allow edges if node has no intra-community edges

        # Iterate all edges incident to this node in the full graph g
        for edge_id in g.incident(node_index_g):
            edge = g.es[edge_id]
            src, dst = g.vs[edge.source], g.vs[edge.target]
            sourceId, targetId = src["id"], dst["id"]
            linkedNode = sourceId if sourceId != node_g["id"] else targetId

            # Check presence in backbone regardless of weight threshold
            src_backbone_idx = id_to_vertex_backbone.get(node_g["id"])
            dst_backbone_idx = id_to_vertex_backbone.get(linkedNode)
            in_backbone = (
                src_backbone_idx is not None and
                dst_backbone_idx is not None and
                backbone_graph.are_connected(src_backbone_idx, dst_backbone_idx)
            )

            # Decide inclusion and status
            include = in_backbone or (edge["weight"] >= node_min - 1e-12)
            if not include:
                continue

            if in_backbone:
                status = "Visible"
            elif node_to_comm[linkedNode] == comm_self:
                status = "Hidden"
            else:
                status = "External"

            edges_info.append((edge["weight"], src["game_name"], dst["game_name"],
                               sourceId, targetId, status, linkedNode))

        # Build pydot edges/nodes
        for weight, source, target, sourceId, targetId, status, linkedNode in sorted(edges_info):
            # Cross-community backbone edges should be Visible in both communities
            if status == "Visible" and node_to_comm[linkedNode] != comm_self:
                ghost_id = str(int(linkedNode + 1_000_000))
                pedge = pydot.Edge(target_node, ghost_id, weight=f"{weight:.10f}")
                pnode = pydot.Node(
                    ghost_id,
                    id=str(int(linkedNode)),
                    c=str(int(node_to_comm[linkedNode]))
                )
                all_results.append((pedge, pnode))

            elif status == "External":
                ghost_id = str(int(linkedNode + 1_000_000))
                pedge = pydot.Edge(target_node, ghost_id, weight=f"{weight:.10f}", s=status)
                pnode = pydot.Node(
                    ghost_id,
                    id=str(int(linkedNode)),
                    c=str(int(node_to_comm[linkedNode]))
                )
                all_results.append((pedge, pnode))

            elif status == "Hidden":
                ghost_id = id_to_name_cache[str(int(linkedNode))]
                pedge = pydot.Edge(target_node, ghost_id, weight=f"{weight:.10f}", s=status)
                all_results.append((pedge, None))

    # Merge into final graph
    for pedge, pnode in all_results:
        if not graph.get_edge(pedge.get_source(), pedge.get_destination()):
            graph.add_edge(pedge)
        if pnode and not graph.get_node(pnode.get_name()):
            graph.add_node(pnode)

    for node in graph.get_nodes():
        
        attrs = node.get_attributes()
        if "game_name" in attrs:
            del node.obj_dict['attributes']['game_name']

    # Ensure output dir exists
    os.makedirs("../extendedGraph", exist_ok=True)

    outpath = f"../extendedGraph/subgraph_{j}.dot"
    with open(outpath, "w") as f:
        f.write(graph.to_string())

    return outpath

# ----------------------------
# Build pydot subgraph from tmfg_backbone_graph by community
# ----------------------------
def pydot_subgraph_from_igraph_by_community(ig_graph, community_label):
    """Create a pydot undirected graph from the vertices in igraph with vertex attribute community == community_label."""
    if "community" not in ig_graph.vs.attributes():
        raise ValueError("tmfg_backbone_graph is missing 'community' vertex attribute.")

    # Collect vertices in the target community
    sub_vs = [v.index for v in ig_graph.vs if v["community"] == community_label]
    idx_map = {old_i: str(new_i) for new_i, old_i in enumerate(sub_vs)}

    pdg = pydot.Dot(graph_type='graph')  # undirected

    # Add nodes with id and label attributes from the original graph
    for new_i, old_i in enumerate(sub_vs):
        v = ig_graph.vs[old_i]
        node_id = int(v["id"]) if "id" in v.attributes() else old_i
        node_label = v["game_name"] if "game_name" in v.attributes() else str(node_id)
        pdg.add_node(pydot.Node(str(new_i), id=str(node_id), label=str(node_label)))

    # Add edges within the community, carrying weight
    sub_set = set(sub_vs)
    for e in ig_graph.es:
        u, v = e.source, e.target
        if u in sub_set and v in sub_set:
            w = float(e["weight"]) if "weight" in e.attributes() else 1.0
            pdg.add_edge(pydot.Edge(idx_map[u], idx_map[v], weight=f"{w:.10f}"))

    return pdg

# ----------------------------
# Parallel over communities (constructed from tmfg_backbone_graph)
# ----------------------------
N_WORKERS = 11  # adjust to your CPU

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {
        executor.submit(
            process_one_graph,
            pydot_subgraph_from_igraph_by_community(tmfg_backbone_graph, int(j)),
            g,
            tmfg_backbone_graph,
            int(j)
        ): int(j)
       for j in sorted({int(c) for c in tmfg_backbone_graph.vs['community'] if c is not None})
    }
    for fut in as_completed(futures):
        j = futures[fut]
        print(f"Finished subgraph {j} → {fut.result()}")

Finished subgraph 51 → ../extendedGraph/subgraph_51.dot
Finished subgraph 53 → ../extendedGraph/subgraph_53.dot
Finished subgraph 18 → ../extendedGraph/subgraph_18.dot
Finished subgraph 46 → ../extendedGraph/subgraph_46.dot
Finished subgraph 52 → ../extendedGraph/subgraph_52.dot
Finished subgraph 50 → ../extendedGraph/subgraph_50.dot
Finished subgraph 30 → ../extendedGraph/subgraph_30.dot
Finished subgraph 13 → ../extendedGraph/subgraph_13.dot
Finished subgraph 41 → ../extendedGraph/subgraph_41.dot
Finished subgraph 29 → ../extendedGraph/subgraph_29.dot
Finished subgraph 27 → ../extendedGraph/subgraph_27.dot
Finished subgraph 16 → ../extendedGraph/subgraph_16.dot
Finished subgraph 40 → ../extendedGraph/subgraph_40.dot
Finished subgraph 39 → ../extendedGraph/subgraph_39.dot
Finished subgraph 42 → ../extendedGraph/subgraph_42.dot
Finished subgraph 20 → ../extendedGraph/subgraph_20.dot
Finished subgraph 49 → ../extendedGraph/subgraph_49.dot
Finished subgraph 22 → ../extendedGraph/subgraph

In [5]:
import os
from igraph import write

# Ensure graph g exists
try:
    g
except NameError:
    raise RuntimeError("Graph 'g' is not defined. Build it before exporting.")

os.makedirs("../graph", exist_ok=True)
output_path = "../graph/boardgames_graph.dot"

# Format edge weights to avoid scientific notation
if 'weight' in tmfg_backbone_graph.es.attributes():
    original_weights = [e['weight'] for e in tmfg_backbone_graph.es]
    tmfg_backbone_graph.es['weight'] = [f"{w:.10f}" for w in original_weights]

# When exporting to DOT, igraph will include vertex attributes like 'pos'
write(tmfg_backbone_graph, output_path, format="dot")
print(f"Written DOT file with positions: {output_path} ({tmfg_backbone_graph.vcount()} nodes, {tmfg_backbone_graph.ecount()} edges)")

# Restore numeric weights for further processing if needed
if 'weight' in tmfg_backbone_graph.es.attributes():
    tmfg_backbone_graph.es['weight'] = original_weights

Written DOT file with positions: ../graph/boardgames_graph.dot (40590 nodes, 84695 edges)


In [6]:
# List all connections for nodes in community 80 in tmfg_backbone_graph
import pandas as pd

TARGET_COMMUNITY = 55
rows = []

# Iterate nodes in the target community and list their neighbors
for v in tmfg_backbone_graph.vs:
    if v["community"] == TARGET_COMMUNITY:
        v_id = v["id"] if "id" in v.attributes() else v.index
        v_label = v["game_name"] if "game_name" in v.attributes() else str(v_id)
        for nbr_idx in tmfg_backbone_graph.neighbors(v.index):
            # Get edge weight if present
            e_id = tmfg_backbone_graph.get_eid(v.index, nbr_idx, directed=False, error=False)
            weight = None
            if e_id != -1:
                e = tmfg_backbone_graph.es[e_id]
                weight = e["weight"] if "weight" in e.attributes() else None
            nbr = tmfg_backbone_graph.vs[nbr_idx]
            nbr_id = nbr["id"] if "id" in nbr.attributes() else nbr_idx
            nbr_label = nbr["game_name"] if "game_name" in nbr.attributes() else str(nbr_id)
            nbr_comm = nbr["community"] if "community" in nbr.attributes() else None
            rows.append({
                "source_id": v_id,
                "source_label": v_label,
                "target_id": nbr_id,
                "target_label": nbr_label,
                "target_community": nbr_comm,
                "weight": weight
            })

# Create DataFrame and sort for readability
connections_df = pd.DataFrame(rows)
if not connections_df.empty:
    connections_df = connections_df.sort_values(["source_label", "target_label"]).reset_index(drop=True)
    display(connections_df)
else:
    print(f"No nodes found in community {TARGET_COMMUNITY} or no connections present.")

,source_id,source_label,target_id,target_label,target_community,weight
0,381436.0,Clash of Decks: Belligerency,381439.0,Clash of Decks: Bogged Down,55.0,0.909091
1,381436.0,Clash of Decks: Belligerency,381440.0,Clash of Decks: Discord,55.0,0.833333
2,381436.0,Clash of Decks: Belligerency,369763.0,Clash of Decks: Elusive,55.0,0.238095
3,381436.0,Clash of Decks: Belligerency,381438.0,Clash of Decks: Slyness,55.0,0.666667
4,381439.0,Clash of Decks: Bogged Down,381436.0,Clash of Decks: Belligerency,55.0,0.909091
5,381439.0,Clash of Decks: Bogged Down,381440.0,Clash of Decks: Discord,55.0,0.909091
6,381439.0,Clash of Decks: Bogged Down,369763.0,Clash of Decks: Elusive,55.0,0.243902
7,381439.0,Clash of Decks: Bogged Down,369764.0,Clash of Decks: Resistance,55.0,0.222222
8,381439.0,Clash of Decks: Bogged Down,381438.0,Clash of Decks: Slyness,55.0,0.714286
9,369760.0,Clash of Decks: Breaching In,367089.0,Clash of Decks: Deliquescence,55.0,0.764706


In [12]:
# Compute quintile thresholds (5 equal-frequency groups) for edge weights in tmfg_backbone_graph (igraph)
import numpy as np
import pandas as pd

# Use the igraph Graph already in memory
_g = tmfg_backbone_graph

# Collect edge weights robustly
edge_weights = None
try:
    edge_weights = _g.es["weight"]
except KeyError:
    # Fallback: use the first numeric edge attribute if 'weight' is absent
    for attr_name in _g.es.attributes():
        values = _g.es[attr_name]
        if len(values) and isinstance(values[0], (int, float, np.floating, np.integer)):
            edge_weights = values
            break

if edge_weights is None or len(edge_weights) == 0:
    print("No numeric edge weights found on tmfg_backbone_graph.")
else:
    arr = np.asarray(edge_weights, dtype=float)
    # Quintiles: 0%, 20%, 40%, 60%, 80%, 100%
    qs = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    q_values = np.quantile(arr, qs)
    
    quintile_thresholds = {f"P{int(q*100)}": v for q, v in zip(qs, q_values)}
    
    thresholds_df = pd.DataFrame({
        "percentile": (qs * 100).astype(int),
        "threshold": q_values
    })
    
    print("Quintile thresholds for edge weights (0%, 20%, ..., 100%):")
    display(thresholds_df)
    
    # Optional: expose mapping for reuse later
    quintile_thresholds

Quintile thresholds for edge weights (0%, 20%, ..., 100%):


,percentile,threshold
0,0,0.000077
1,20,0.011184
2,40,0.046948
3,60,0.080745
4,80,0.142361
5,100,1.000000


In [13]:
# Show distribution of edges across quintiles (for link colors)
import pandas as pd

edge_weights = np.asarray(tmfg_backbone_graph.es["weight"], dtype=float)

# Create bins using quintile thresholds
bins = [0, 0.011184, 0.046948, 0.080745, 0.142361, 1.0]
labels = ['#4a148c', '#7b1fa2', '#ab47bc', '#ff7043', '#ff5722']

# Assign each edge to a quintile
edge_quintiles = pd.cut(edge_weights, bins=bins, labels=labels, include_lowest=True)

# Count edges per quintile
quintile_edge_counts = edge_quintiles.value_counts().sort_index()

print("Edges per quintile (link color):")
display(quintile_edge_counts)

print(f"\nTotal edges: {len(edge_weights)}")
print(f"Expected per quintile: ~{len(edge_weights) // 5}")

Edges per quintile (link color):


#4a148c    16940
#7b1fa2    16926
#ab47bc    16950
#ff7043    16939
#ff5722    16940
Name: count, dtype: int64


Total edges: 84695
Expected per quintile: ~16939


In [10]:
# Compute rating intervals (10 equal-frequency groups - deciles) for games in graph g
import numpy as np
import pandas as pd

# Load game data
games_df = pd.read_csv("../data/bgg_GameItem(4).csv")

# Get game IDs from graph g
game_ids_in_graph = [v["id"] for v in g.vs if "id" in v.attributes()]

# Filter to only games present in the graph
games_in_graph = games_df[games_df["bgg_id"].isin(game_ids_in_graph)].copy()

# Get ratings for games in graph (drop NaN ratings if any)
ratings = games_in_graph["avg_rating"].dropna().values

print(f"Total games in graph: {len(game_ids_in_graph)}")
print(f"Games with ratings: {len(ratings)}")
print()

# 10 intervals (deciles): 0%, 10%, 20%, ..., 90%, 100%
qs = np.array([i/10 for i in range(11)])
q_values = np.quantile(ratings, qs)

rating_interval_thresholds = {f"P{int(q*100)}": v for q, v in zip(qs, q_values)}

thresholds_df = pd.DataFrame({
    "percentile": (qs * 100).round(2),
    "threshold": q_values
})

print("Rating interval thresholds (10 equal-frequency groups - deciles):")
display(thresholds_df)

# Assign each game to its rating interval bin (1-10)
games_in_graph["rating_interval"] = pd.cut(
    games_in_graph["avg_rating"],
    bins=q_values,
    labels=range(1, 11),
    include_lowest=True,
    duplicates='drop'
)

# Show distribution of games across intervals
interval_counts = games_in_graph["rating_interval"].value_counts().sort_index()
print("\nGames per rating interval:")
display(interval_counts)

# Optional: expose mapping for reuse later
rating_interval_thresholds


Total games in graph: 40590
Games with ratings: 40567

Rating interval thresholds (10 equal-frequency groups - deciles):


/tmp/ipykernel_3550/2280933492.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  games_df = pd.read_csv("../data/bgg_GameItem(4).csv")


,percentile,threshold
0,0.0,1.183330
1,10.0,5.076920
2,20.0,5.565964
3,30.0,5.900366
4,40.0,6.181570
5,50.0,6.422970
6,60.0,6.659090
7,70.0,6.918946
8,80.0,7.204706
9,90.0,7.609174



Games per rating interval:


rating_interval
1     4059
2     4055
3     4056
4     4057
5     4057
6     4058
7     4055
8     4056
9     4057
10    4057
Name: count, dtype: int64

{'P0': 1.18333,
 'P10': 5.07692,
 'P20': 5.565964,
 'P30': 5.900366,
 'P40': 6.18157,
 'P50': 6.42297,
 'P60': 6.65909,
 'P70': 6.918946,
 'P80': 7.204706,
 'P90': 7.609174,
 'P100': 10.0}